In [1]:
import Fase_2v2_GMMHMMAligner as fase2
from Fase_2v2_GMMHMMAligner import *
import Fase_1_CreateAligner as fase1
from Fase_1_CreateAligner import *

In [ ]:
import pandas as pd

CSV_IN  = r"D:\118_Rio\118\src\codeStat\preModelling\out\databersih.csv"
CSV_OUT = CSV_IN  # overwrite

OLD = r"D:\doc\2026\FINAL_CODE\data\sampledData"
NEW = r"D:\118_Rio\118\sampled"

df = pd.read_csv(CSV_IN)
df["Path_Data"] = df["Path_Data"].str.replace(OLD, NEW, regex=False)
df.to_csv(CSV_OUT, index=False, encoding="utf-8")
print(f"Selesai — {len(df):,} baris diperbarui")
print(df["Path_Data"].iloc[0])  # verifikasi

Selesai — 17,880 baris diperbarui
D:\118_Rio\118\sampled\0\00\00e1f837f4.wav


In [9]:
import os, shutil
import pandas as pd

CSV_PATH   = r"D:\118_Rio\118\src\codeStat\preModelling\out\databersih.csv"
DATA_ROOT  = r"D:\118_Rio\118\data"   # folder berkas suara asli
DEST_ROOT  = r"D:\118_Rio\118\datasampled"  # folder tujuan baru

EXT = ".wav"  # ganti ke .flac jika perlu


def kumpulkan_audio(csv_path: str, data_root: str, dest_root: str) -> None:
    df = pd.read_csv(csv_path)
    print(f"  Total baris CSV     : {len(df):,}")

    # Bangun indeks: stem → path lengkap dari folder Data
    print("  Mengindeks folder Data ...")
    indeks = {}
    for root, _, files in os.walk(data_root):
        for fname in files:
            if fname.lower().endswith(EXT):
                stem = os.path.splitext(fname)[0]
                indeks[stem] = os.path.join(root, fname)
    print(f"  File terindeks      : {len(indeks):,}")

    # Cocokkan stem dari CSV dengan indeks
    os.makedirs(dest_root, exist_ok=True)
    cocok, tidak_cocok = 0, 0

    for _, row in df.iterrows():
        stem = os.path.splitext(os.path.basename(str(row["Path_Data"])))[0]
        src  = indeks.get(stem)

        if src:
            dest = os.path.join(dest_root, os.path.basename(src))
            shutil.copy2(src, dest)
            cocok += 1
        else:
            tidak_cocok += 1

    print(f"  Berhasil disalin    : {cocok:,}")
    print(f"  Tidak ditemukan     : {tidak_cocok:,}")
    print(f"  Output              : {dest_root}")


def main():
    print("Pengumpulan Berkas Audio ke datasampled")
    kumpulkan_audio(CSV_PATH, DATA_ROOT, DEST_ROOT)


if __name__ == "__main__":
    main()

Pengumpulan Berkas Audio ke datasampled
  Total baris CSV     : 17,880
  Mengindeks folder Data ...
  File terindeks      : 185,076
  Berhasil disalin    : 17,880
  Tidak ditemukan     : 0
  Output              : D:\118_Rio\118\datasampled


In [2]:
import sys
sys.argv = ["mfa_script.py", "--prepare"]  # ← bukan --mode prepare
fase1.main()

  FASE 1 — Pembuatan Word Alignment dengan MFA
  Transkripsi dimuat: 17880 audio

Menyiapkan folder korpus MFA ...
Mempersiapkan 17880 audio untuk MFA ...
Selesai: 17880 audio siap, 0 dilewati (tidak ada transkripsi)
Folder korpus: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_workspace\corpus

Membuat pronunciation dictionary ...
Dictionary: 20144 kata → D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_workspace\javanese_dictionary.txt

  LANGKAH SELANJUTNYA — Jalankan di Anaconda Prompt

  1. Aktifkan environment MFA:
     conda activate mfa

  2. Latih model akustik sekaligus alignment:
     mfa train "D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_workspace\corpus" "D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_workspace\javanese_dictionary.txt" "D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sam

In [3]:
sys.argv = ["mfa_script.py", "--validate"]  # ← bukan --mode prepare
fase1.main()

  FASE 1 — Pembuatan Word Alignment dengan MFA
  Transkripsi dimuat: 17880 audio

Membaca hasil MFA ...
Memproses 17858 file TextGrid ...

  ── Ringkasan Alignment ──────────────────────────
  TextGrid berhasil  : 17858 (99.9% dari total audio)
  TextGrid kosong    : 0
  Interval difilter  : 22 (< 3 frame = noise)
  Total kata-interval: 123944
  Kata unik          : 20127
  Audio unik         : 17858
  Output             : D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\makeAligner\sampleddata\alignment_map.csv
  ────────────────────────────────────────────────

Validasi hasil alignment ...

  Statistik durasi per kata (1 frame = 10ms):
    Min    : 3 frame
    Median : 47 frame
    Rerata : 50.9 frame
    Max    : 634 frame

  Kata unik dengan alignment : 20127
  Kata dengan ≥5 kemunculan  : 4062
  Kata dengan 1 kemunculan   : 10391 (kemungkinan OOV)

  Alignment siap digunakan oleh Fase 2.
